# RAG from Scratch to Production — Episode 1/20
## Why RAG? Watching an LLM fail on *your* data, then fixing it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/01_why_rag.ipynb)

This series builds up, concept by concept, the [RAG chatbot](https://github.com/JakkulaMahendhar/rag-chatbot) I built: FastAPI + ChromaDB + BM25 + CrossEncoder reranking + Gemini/Ollama.

**In this episode:**
1. Ask an LLM about a private company document it has never seen, and watch it fail.
2. Try the obvious fix ("just paste the whole document in") and see why it doesn't scale.
3. Build a **20-line RAG pipeline** (chunk → retrieve → augment → generate) that answers correctly *and* refuses when it doesn't know.
4. Break it on purpose, which sets up Episode 2.

No vector DB and no embeddings yet. Just plain Python, so the *idea* is clear before the tooling arrives.

## 0. Setup: pick an LLM

Same idea as `app/services/llm/` in the [production app](https://github.com/JakkulaMahendhar/rag-chatbot): one `generate(prompt) -> str` function, two providers behind it.

- **Gemini** (easiest in Colab): get a free key at https://aistudio.google.com/apikey and add it as a Colab secret named `GEMINI_API_KEY`.
- **Ollama** (fully local): `ollama pull llama3.1`, then set `PROVIDER = "ollama"`.

In [ ]:
%pip install -q google-generativeai requests

In [2]:
import os, requests

PROVIDER = os.getenv("LLM_PROVIDER", "gemini")   # "gemini" or "ollama"
GEMINI_MODEL = "gemini-3.6-flash"
OLLAMA_MODEL = "llama3.1"


def _gemini_key():
    try:
        from google.colab import userdata  # Colab secrets
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        return os.getenv("GEMINI_API_KEY") or __import__("getpass").getpass("GEMINI_API_KEY: ")


if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=_gemini_key())
    _model = genai.GenerativeModel(GEMINI_MODEL)


def generate(prompt: str) -> str:
    """One interface, two providers. Temperature 0 so results are repeatable."""
    if PROVIDER == "gemini":
        return _model.generate_content(
            prompt, generation_config={"temperature": 0}
        ).text.strip()
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
              "options": {"temperature": 0}},
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["response"].strip()


print(f"Using {PROVIDER}")

Using ollama


## 1. The private document

A (fictional) employee handbook. This is the kind of data an LLM **was never trained on**: internal, recent, specific to your company.

In [3]:
HANDBOOK = """
# NovaTech Solutions — Employee Handbook (2026 edition)

## Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory
in-office day for all teams. Remote days must be logged in the HR portal by
Friday of the previous week.

## Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year.
A maximum of 5 unused days may be carried over into the next year; the rest
expire on 31 March.

## Expenses
Business expenses must be submitted within 30 days with a receipt. The meal
allowance during business travel is capped at 40 USD per day.

## Equipment
Every employee receives a laptop, refreshed every 3 years. Lost or stolen
devices must be reported to IT within 24 hours.

## Learning Budget
Each employee has an annual learning budget of 1,200 USD for courses,
certifications, and conferences. Unused budget does not roll over.

## Security
Multi-factor authentication is mandatory on all company accounts. Suspected
security incidents must be reported to security@novatech.example within 1 hour.
""".strip()

print(f"{len(HANDBOOK.split())} words")

174 words


## 2. Ask the LLM directly (no context)

In [4]:
question = "What is the annual learning budget per employee at NovaTech Solutions, and does unused budget roll over?"

print(generate(question))

Unfortunately, I don't have access to specific information about NovaTech Solutions' policies or budget allocations. However, I can provide some general insights on the concept of an annual learning budget and how it might work in a company like NovaTech Solutions.

In many organizations, an annual learning budget is allocated to each employee to support their professional development and growth. This budget can be used to fund various learning activities, such as:

*   Online courses and training programs
*   Conferences and workshops
*   Books and other educational resources
*   Mentorship programs
*   Coaching or counseling services

The annual learning budget per employee can vary widely depending on factors such as:

*   Company size and industry
*   Employee level and role
*   Company culture and values
*   Budget allocation and priorities

Regarding the question of whether unused budget rolls over, it's common for companies to have policies in place to handle unused learning bud

The correct answer is **1,200 USD, and it does not roll over**.

A bare LLM can only do one of two things here:
- **Hallucinate:** make up a confident, plausible number.
- **Deflect:** "I don't have access to NovaTech's policies… but here's how learning budgets *generally* work…", followed by paragraphs of generic filler.

Different models (and runs) do different things, but neither answer helps an employee. The model isn't broken. It just **never saw this document**. Its knowledge is frozen at training time and never included your private data.

## 3. The obvious fix: paste the whole document into the prompt

For one small handbook, this works:

In [5]:
stuffed_prompt = f"""Answer using this document:

{HANDBOOK}

Question: {question}"""

print(generate(stuffed_prompt))

According to the document, each employee has an annual learning budget of 1,200 USD for courses, certifications, and conferences. Unused budget does not roll over.


It works. So why doesn't everyone just do this? Scale it to a real company:

In [6]:
def approx_tokens(text: str) -> int:
    return len(text) // 4          # rule of thumb: ~4 characters per token

handbook_tokens = approx_tokens(HANDBOOK)

# A real company knowledge base
docs, pages_per_doc, tokens_per_page = 5_000, 20, 500
kb_tokens = docs * pages_per_doc * tokens_per_page

print(f"This handbook          : {handbook_tokens:>12,} tokens")
print(f"Company knowledge base : {kb_tokens:>12,} tokens")
print(f"Largest context windows: {2_000_000:>12,} tokens (approx.)")
print(f"\nThe knowledge base is {kb_tokens / 2_000_000:.0f}x bigger than even a huge context window,")
print("and you would be paying for all of those tokens on every single question.")

This handbook          :          263 tokens
Company knowledge base :   50,000,000 tokens
Largest context windows:    2,000,000 tokens (approx.)

The knowledge base is 25x bigger than even a huge context window,
and you would be paying for all of those tokens on every single question.


Stuffing everything into the prompt fails on:

| Problem | Why it hurts |
|---|---|
| **Size** | Real knowledge bases are far larger than any context window |
| **Cost & latency** | You pay for (and wait on) every token, on every question |
| **Accuracy** | Models get worse at finding facts buried in very long prompts ("lost in the middle") |
| **Access control** | You'd be sending *every* document to *every* user |

What we actually want: **send only the few passages that are relevant to this question.**

## 4. RAG in 20 lines

**R**etrieval-**A**ugmented **G**eneration has four steps:

```
 documents ──► 1. CHUNK ──► chunks
 question  ──► 2. RETRIEVE the most relevant chunks
           ──► 3. AUGMENT the prompt with only those chunks
           ──► 4. GENERATE a grounded answer
```

### Step 1: Chunk
Split the document into small, self-contained pieces. Here each `##` section is one chunk. (Episode 4 covers real chunking strategies; the app's version is `app/services/chunker.py`.)

In [7]:
chunks = [c.strip() for c in HANDBOOK.split("\n## ")[1:]]

for i, c in enumerate(chunks):
    print(i, "→", c.splitlines()[0])

0 → Remote Work
1 → Annual Leave
2 → Expenses
3 → Equipment
4 → Learning Budget
5 → Security


### Step 2: Retrieve
Score every chunk against the question and keep the best one(s). For now, the simplest possible scorer is **how many meaningful words the question and the chunk share**. (Episodes 5–10 replace this with embeddings, BM25, hybrid search, and a reranker.)

In [8]:
import re

STOPWORDS = {"a", "an", "the", "is", "are", "and", "or", "of", "to", "in", "on",
             "at", "for", "per", "what", "how", "many", "much", "does", "do",
             "i", "my", "can", "get", "it", "be", "must", "by", "with"}

def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}

def retrieve(question: str, chunks: list[str], top_k: int = 1):
    q = words(question)
    scored = [(len(q & words(c)), c) for c in chunks]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [(s, c) for s, c in scored[:top_k] if s > 0]

for score, chunk in retrieve(question, chunks):
    print(f"score={score}\n{chunk}")

score=7
Learning Budget
Each employee has an annual learning budget of 1,200 USD for courses,
certifications, and conferences. Unused budget does not roll over.


### Step 3: Augment
Put **only the retrieved chunk** into a prompt with strict grounding rules. This is a trimmed version of `app/services/prompt_builder.py`:

In [9]:
FALLBACK = "I don't have enough information from the provided documents."

def build_prompt(question: str, context: str) -> str:
    return f"""You are an assistant that answers questions from provided documents.

Rules:
1. Answer only using the context below.
2. Do not use external knowledge or assume facts.
3. If the answer is not in the context, respond exactly: "{FALLBACK}"

Context:
{context or "(no relevant context found)"}

Question: {question}
Answer:"""

### Step 4: Generate

In [10]:
def rag(question: str, top_k: int = 1) -> str:
    hits = retrieve(question, chunks, top_k)
    context = "\n\n".join(c for _, c in hits)
    print("Retrieved:", [c.splitlines()[0] for c in [h for _, h in hits]] or "nothing")
    return generate(build_prompt(question, context))

print(rag(question))

Retrieved: ['Learning Budget']


According to the provided context, the annual learning budget per employee at NovaTech Solutions is 1,200 USD. Additionally, it is stated that "Unused budget does not roll over."


The answer is correct, it's grounded in the retrieved chunk, and the prompt was a fraction of the size.

### The underrated half of RAG: knowing when *not* to answer

In [11]:
print(rag("What is NovaTech's parental leave policy?"))

Retrieved: ['Annual Leave']


I don't have enough information from the provided documents.


Look closely: retrieval returned the **Annual Leave** chunk (it matched the word "leave"), which is a near miss. The strict prompt rules still stopped the model from stretching annual-leave rules into a parental-leave policy, so it **refuses instead of inventing one**. In production, a confident wrong answer about HR or legal policy is far worse than "I don't know". (Episode 15 adds a hallucination guard on top of this.)

## 5. Now let's break it (cliffhanger for Episode 2)

Employees don't talk like handbooks. Ask the same thing using everyday words:

In [12]:
casual = "How often am I allowed to WFH?"

for score, chunk in retrieve(casual, chunks, top_k=3):
    print(f"score={score} → {chunk.splitlines()[0]}")

print()
print(rag(casual))


Retrieved: nothing


I don't have enough information from the provided documents.


The answer is **right there** in the Remote Work section, yet the bot says it doesn't know.

The user said **"WFH"** and **"allowed"**. The document says **"remote"** and **"may"**. Same meaning, zero shared words, so keyword retrieval finds nothing and the model (correctly!) refuses. The LLM did its job. **Retrieval failed it.**

**Retrieval quality is the ceiling on answer quality.** The LLM can only be as good as the context you give it. The rest of this series is mostly about getting retrieval right:

| Episode | Upgrade | File in the app |
|---|---|---|
| 4 | Real chunking (size + overlap) | `app/services/chunker.py` |
| 5–6 | Embeddings + a vector DB, so "WFH" ≈ "remote" | `embedding.py`, `chroma_client.py` |
| 7–8 | BM25 + hybrid search | `bm25_search.py`, `hybrid_search.py` |
| 10 | CrossEncoder reranking | reranker |
| 15 | Hallucination guard | `hallucination_guard_service.py` |

## Recap

| Approach | Private data? | Scales? | Refuses when unsure? |
|---|---|---|---|
| Plain LLM | ❌ | — | ❌ often hallucinates |
| Paste everything | ✅ | ❌ | ⚠️ |
| **RAG** | ✅ | ✅ | ✅ |

**RAG = give the model an open-book exam, but hand it only the right page.**

**Next, Episode 2:** the full architecture of a production RAG system, following one request end to end through the real app.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch  
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot  
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)